# Jev hands-on

Jev (TypeSafe) does not write a reply. You send a **state** and typed **questions**. One call returns probabilities.

Three question types, from the [TypeSafe API](https://docs.typesafe.ai/api):

- `choice` — pick one option you named (max 255)
- `score` — place the state on an ordered scale (2–10 levels)
- `noul` — yes/no as a probability from 0 to 1

Key: [console.typesafe.ai](https://console.typesafe.ai) → `TYPESAFE_API_KEY`. Jev is still early access, so a new account may be waitlisted.

Same exercises on Drex: [drex_hands_on.ipynb](drex_hands_on.ipynb).

In [ ]:
import os

os.environ["TYPESAFE_API_KEY"] = ""  # paste sk key; do not commit it

API_KEY = os.environ["TYPESAFE_API_KEY"].strip()
ENDPOINT = "https://api.typesafe.ai/v1/systemone"
MODEL = "jev-latest"

assert API_KEY, "Set TYPESAFE_API_KEY in this cell"

## One call

`POST /v1/systemone` with `state`, `model`, and `questions`. Questions in one request share the state and run in parallel.

In [ ]:
import json
import urllib.request


def decide(state, questions):
    body = json.dumps({"model": MODEL, "state": state, "questions": questions}).encode()
    req = urllib.request.Request(
        ENDPOINT,
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.load(resp)


TICKET = (
    "Customer: I was charged twice last Tuesday and I am furious. "
    "Order #4419. I want a refund today."
)

QUESTIONS = {
    "topic": {
        "type": "choice",
        "instructions": "What is the issue about?",
        "criteria": {
            "billing": "Charges, refunds, invoices",
            "bug": "Something in the product is broken",
            "shipping": "Delivery or a missing order",
        },
    },
    "frustration": {
        "type": "score",
        "instructions": "How frustrated does the customer sound?",
        "criteria": ["Calm", "Annoyed", "Furious"],
    },
    "escalate": {
        "type": "noul",
        "instructions": "Escalate to a human now?",
    },
}

result = decide(TICKET, QUESTIONS)
print("model:", result["model"])
print("tokens:", result["usage"])
print(json.dumps(result["answers"], indent=2))

## Act on the probabilities

Your code owns the threshold. A high `noul` escalates. Otherwise route to the winning `choice`.

In [ ]:
answers = result["answers"]
topic = answers["topic"]
mood = answers["frustration"]
escalate_p = answers["escalate"]["noul"]

print("topic:", topic["choice"], topic["probabilities"])
print("frustration score:", mood["score"], "legend:", mood["legend"])
print("escalate probability:", escalate_p)

if escalate_p >= 0.8:
    action = "send to a human"
else:
    action = f"route to {topic['choice']}"
print("action:", action)

## Your turn

Change `state` and rerun. Keep the options closed. If the real answer is not in `criteria`, Jev still has to pick one of them.

In [ ]:
mine = decide(
    "The export button crashes Safari. No workaround. Customers cannot download invoices.",
    {
        "team": {
            "type": "choice",
            "instructions": "Which team should take this?",
            "criteria": {
                "billing": "Money movement",
                "engineering": "Product bug",
                "support": "How-to, no code change",
            },
        },
        "blocking": {
            "type": "noul",
            "instructions": "Is a core workflow blocked?",
        },
    },
)
print(json.dumps(mine["answers"], indent=2))